### ONE HOT ENCODING OF WORDS OR CHARACTERS

#### Word level one-hot encoding

In [1]:
import numpy as np

# sample sentence
samples = ['The cat sat on the mat.', 'The dog ate my homework.']

In [4]:
# build an indec of all tokens in the samples
token_index = {} # create empty dictionary
for sample in samples:
    # we do with split method
    for word in sample.split():
        if word not in token_index:
            token_index[word] = len(token_index) + 1

In [5]:
# vectorize the samples
# with max length
max_length = 10

In [6]:
# the results
results = np.zeros((len(samples), max_length, max(token_index.values()) + 1))

for i, sample in enumerate(samples):
    for j, word in list(enumerate(sample.split()))[:max_length]:
        index = token_index.get(word)
        results[i , j, index] = 1

#### Using keras for word-level one-hot encoding

In [11]:
# use tokenizer from keras
from tensorflow.keras.preprocessing.text import Tokenizer

In [12]:
# create samples
samples = ['The cat sat on the mat.', 'The dog ate my homework.']

In [13]:
# create tokenizer with only take 1000 most common words
tokenizer = Tokenizer(num_words=1000)

In [14]:
# fit on text
tokenizer.fit_on_texts(samples)

In [17]:
# get the word index
word_index = tokenizer.word_index
word_index

{'the': 1,
 'cat': 2,
 'sat': 3,
 'on': 4,
 'mat': 5,
 'dog': 6,
 'ate': 7,
 'my': 8,
 'homework': 9}

In [18]:
# turn strings into list of integer
sequences = tokenizer.texts_to_sequences(samples)
sequences

[[1, 2, 3, 4, 1, 5], [1, 6, 7, 8, 9]]

### USING EMBEDDING LAYER AND CLASSIFIER ON THE IMDB DATA

LOAD DATA

In [3]:
from keras.datasets import imdb
from keras import preprocessing

In [4]:
# number of words
max_feature = 10000

# cut texts after this number of words
maxlen = 20

In [5]:
# load data
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=max_feature)

In [6]:
# turn the data from integers into a 2D integer tensor in shape (samples, maxlen)
x_train = preprocessing.sequence.pad_sequences(x_train, maxlen=maxlen)
x_test = preprocessing.sequence.pad_sequences(x_test, maxlen=maxlen)

NETWORK

In [2]:
from keras.models import Sequential
from keras.layers import Flatten, Dense, Embedding

In [7]:
model = Sequential()

# specify the maximum input length to our embedding layer, so at next we can later flatten the embedded input
model.add(Embedding(10000, 8, input_length=maxlen))
# after the embedding layer, our activations have shape (samples, maxlen, 8)

c:\Users\LENOVO\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\core\embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [8]:
# flatten the 3D tensors of embedding, into 2d tensors shape (samples, maxlen * 8)
model.add(Flatten())

In [9]:
# add FC layer or Classifier on top
model.add(Dense(1, activation='sigmoid'))

# compile model
model.compile(optimizer='rmsprop', loss='binary_crossentropy', metrics=['acc'])

In [10]:
# see the model summary
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [11]:
# train the model
history = model.fit(x_train, y_train, epochs=10, batch_size=32, validation_split=0.2)

Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - acc: 0.5876 - loss: 0.6827 - val_acc: 0.6954 - val_loss: 0.6101
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - acc: 0.7406 - loss: 0.5645 - val_acc: 0.7298 - val_loss: 0.5261
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - acc: 0.7812 - loss: 0.4739 - val_acc: 0.7438 - val_loss: 0.5036
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - acc: 0.8038 - loss: 0.4303 - val_acc: 0.7500 - val_loss: 0.4960
Epoch 5/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - acc: 0.8236 - loss: 0.3958 - val_acc: 0.7524 - val_loss: 0.4976
Epoch 6/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - acc: 0.8317 - loss: 0.3802 - val_acc: 0.7530 - val_loss: 0.5017
Epoch 7/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - acc: 0.8434 - loss: 0.3595 - val_acc: 0.7516 - val_loss: 0.5085
Epoch 8/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - acc: 0.8606 - loss: 0.3333 - val_acc: 0.7488 - val_loss: 0.5136
Epoch 9/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - ac